### Compile table of supported readers

- ProtData (CZI, https://github.com/czbiohub-sf/protdata)
- MSMU (Huh lab, https://github.com/bertis-informatics/msmu)
- ProteoPy (Bludau Lab, https://github.com/UKHD-NP/proteopy)
- alphapepttools (Mann Lab, https://github.com/MannLabs/alphapepttools)

### Manual check (not for publication):

Reader table section for manuscript:

start in reader_table.ipynb (alphapepttools-manuscript) and tally readers for existing packages; justify listing:

- ProtData: io module basic readers, without extensive standardization
    - read_diann reads proteins
    - fragpipe loader reads proteins
    - maxquant_loader reads proteins
    - mztab_loader reads proteins
    - spectronaut_loader reads proteins but has a swappable index in the pivot operation, meaning it should be able to operate on precursor quantities as well
- MSMU: _read_write module, but orient on _reader_registry.py, which returns mudata —> important difference: we don't simply assemble MuData instances but construct linked data objects via MuLink.
    - _delpi.py reads the main psm table and pivots it; no specific reader for the optional protein table https://github.com/bertis-informatics/delpi
    - read_sage reads tmt or lfq output
    - read_diann reads main psm report; mentions explicitly that protein group reader is not implemented yet
    - read_fragpipe  reads main psm table for tmt and lfq experiments; no protein level report
    - read_maxquant reads main psm report for tmt and lfq experiments and seems to have a MaxDiaReader
- ProteoPy:
    - read/diann.py seems to read and pivot diann and optionally sum precursors into proteins? Do they know about PG.MaxLFQ?

In [3]:
alphadia = "AlphaDIA"
alphapept = "AlphaPept"
maxquant = "MaxQuant"
diann = "DIA-NN"
fragpipe = "FragPipe"
spectronaut = "Spectronaut"
mztab = "mzTab"
sage = "Sage"
delpi = "DELPI"
cptac = "CPTAC"
proteomediscoverer = "Proteome Discoverer"
msfragger = "MSFragger"
pfind = "pFind"
openswath = "OpenSWATH"
spectronaut_speclib = "Spectronaut speclib"
directlfq = "DirectLFQ"

In [4]:
# Reader dict without acquisition mode annotation
reader_dict = {
    "ProtData": {
        "psm": [spectronaut],
        "pg": [maxquant, diann, fragpipe, spectronaut, mztab],
        "fragments": [],
    },
    "MSMU": {
        "psm": [sage, maxquant, fragpipe, diann, delpi],
        "pg": [],
        "fragments": [],
    },
    "ProteoPy": {
        "psm": [diann], 
        "pg": [],
        "fragments": [],
    },
    "AlphaPeptTools": {
        "psm": [alphadia, maxquant, diann, alphapept, spectronaut, msfragger, sage, pfind, openswath, spectronaut_speclib],
        "pg": [alphadia, maxquant, diann, alphapept, spectronaut, fragpipe, directlfq, mztab, proteomediscoverer],
        "fragments": [],
    },
}

# Table for LaTeX: four rows (one per package), two columns (PSM and PG readers), with line breaks between packages
def reader_dict_to_latex_table(reader_dict):
    latex = "\\begin{tabular}{l|l|l}\n"
    latex += "Package & PSM Readers & PG Readers \\\\\n"
    latex += "\\hline\n"
    for package, readers in reader_dict.items():
        psm_readers = ", ".join(readers["psm"])
        pg_readers = ", ".join(readers["pg"])
        latex += f"{package} & {psm_readers} & {pg_readers} \\\\\n"
    latex += "\\end{tabular}"
    return latex

def reader_dict_to_markdown_table(reader_dict, empty_placeholder="—"):
    md = "| Package | PSM Readers | PG Readers |\n"
    md += "|---|---|---|\n"
    for package, readers in reader_dict.items():
        psm_readers = ", ".join(readers["psm"]) or empty_placeholder
        pg_readers = ", ".join(readers["pg"]) or empty_placeholder
        md += f"| {package} | {psm_readers} | {pg_readers} |\n"
    return md

def reader_dict_to_dataframe(reader_dict):
    import pandas as pd
    data = []
    for package, readers in reader_dict.items():
        psm_readers = ", ".join(readers["psm"])
        pg_readers = ", ".join(readers["pg"])
        data.append({"Package": package, "PSM Readers": psm_readers, "PG Readers": pg_readers})
    return pd.DataFrame(data)

# Generate tables
reader_dict_to_latex_table(reader_dict)
print(reader_dict_to_markdown_table(reader_dict))
reader_dict_to_dataframe(reader_dict)

| Package | PSM Readers | PG Readers |
|---|---|---|
| ProtData | Spectronaut | MaxQuant, DIA-NN, FragPipe, Spectronaut, mzTab |
| MSMU | Sage, MaxQuant, FragPipe, DIA-NN, DELPI | — |
| ProteoPy | DIA-NN | — |
| AlphaPeptTools | AlphaDIA, MaxQuant, DIA-NN, AlphaPept, Spectronaut, MSFragger, Sage, pFind, OpenSWATH, Spectronaut speclib | AlphaDIA, MaxQuant, DIA-NN, AlphaPept, Spectronaut, FragPipe, DirectLFQ, mzTab, Proteome Discoverer |



,Package,PSM Readers,PG Readers
0,ProtData,Spectronaut,"MaxQuant, DIA-NN, FragPipe, Spectronaut, mzTab"
1,MSMU,"Sage, MaxQuant, FragPipe, DIA-NN, DELPI",
2,ProteoPy,DIA-NN,
3,AlphaPeptTools,"AlphaDIA, MaxQuant, DIA-NN, AlphaPept, Spectro...","AlphaDIA, MaxQuant, DIA-NN, AlphaPept, Spectro..."


In [6]:
# Reader dict with acquisition mode annotation
# Classification is source-verified across protdata/, msmu/, proteopy/, and alphabase/ (alphapepttools delegates to alphabase).
# Notes:
#   - MSMU's MaxDiaReader exists in _maxquant.py but is NOT imported in _reader_registry.py (no public access) -> excluded.
#   - MSMU's DiannProteinGroupReader raises NotImplementedError -> no MSMU PG readers.
#   - ProtData's maxquant/fragpipe/mztab PG loaders are format-agnostic; listed in both DIA and DDA columns.
#   - alphabase's MaxQuant reader only handles msms.txt (DDA); no MaxDIA reader is registered.
#   - alphabase's MSFragger and Sage readers ingest mode-transparent TSV outputs that the tools produce in both DDA and DIA workflows.
reader_mode_dict = {
    "ProtData": {
        "DIA": {
            "psm": [spectronaut],
            "pg": [maxquant, diann, fragpipe, spectronaut, mztab],
            "fragments": [],
        },
        "DDA": {
            "psm": [],
            "pg": [maxquant, fragpipe, mztab],
            "fragments": [],
        },
    },
    "MSMU": {
        "DIA": {
            "psm": [diann, delpi],
            "pg": [],
            "fragments": [],
        },
        "DDA": {
            "psm": [maxquant, sage, fragpipe],
            "pg": [],
            "fragments": [],
        },
    },
    "ProteoPy": {
        "DIA": {
            "psm": [diann],
            "pg": [],
            "fragments": [],
        },
        "DDA": {
            "psm": [],
            "pg": [],
            "fragments": [],
        },
    },
    "AlphaPeptTools": {
        "DIA": {
            "psm": [alphadia, diann, spectronaut, openswath, spectronaut_speclib, msfragger, sage],
            "pg": [alphadia, diann, spectronaut, fragpipe, directlfq, mztab],
            "fragments": [],
        },
        "DDA": {
            "psm": [maxquant, alphapept, msfragger, sage, pfind],
            "pg": [maxquant, alphapept, fragpipe, directlfq, mztab, proteomediscoverer],
            "fragments": [],
        },
    },
}


def reader_mode_dict_to_latex_table(reader_mode_dict, empty_placeholder="---"):
    """LaTeX tabularx with booktabs styling. Nested header (PSM / Protein-group) over DIA / DDA.

    Matches the structure of the existing manuscript table: tabularx with \\textwidth,
    \\toprule / \\midrule / \\bottomrule, and a \\midrule between every package row.
    \\cite{...} keys are not emitted here — add them manually after pasting, as with the
    original generator.
    """
    lines = [
        "\\begin{tabularx}{\\textwidth}{l X X X X}",
        "\\toprule",
        " & \\multicolumn{2}{c}{PSM readers} & \\multicolumn{2}{c}{Protein-group readers} \\\\",
        "\\cmidrule(lr){2-3} \\cmidrule(lr){4-5}",
        "Package & DIA & DDA & DIA & DDA \\\\",
        "\\midrule",
    ]
    packages = list(reader_mode_dict.items())
    for i, (package, modes) in enumerate(packages):
        psm_dia = ", ".join(modes["DIA"]["psm"]) or empty_placeholder
        psm_dda = ", ".join(modes["DDA"]["psm"]) or empty_placeholder
        pg_dia = ", ".join(modes["DIA"]["pg"]) or empty_placeholder
        pg_dda = ", ".join(modes["DDA"]["pg"]) or empty_placeholder
        lines.append(f"{package} & {psm_dia} & {psm_dda} & {pg_dia} & {pg_dda} \\\\")
        if i < len(packages) - 1:
            lines.append("\\midrule")
    lines.append("\\bottomrule")
    lines.append("\\end{tabularx}")
    return "\n".join(lines)


def reader_mode_dict_to_markdown_table(reader_mode_dict, empty_placeholder="—"):
    md = "| Package | PSM-DIA | PSM-DDA | PG-DIA | PG-DDA |\n"
    md += "|---|---|---|---|---|\n"
    for package, modes in reader_mode_dict.items():
        psm_dia = ", ".join(modes["DIA"]["psm"]) or empty_placeholder
        psm_dda = ", ".join(modes["DDA"]["psm"]) or empty_placeholder
        pg_dia = ", ".join(modes["DIA"]["pg"]) or empty_placeholder
        pg_dda = ", ".join(modes["DDA"]["pg"]) or empty_placeholder
        md += f"| {package} | {psm_dia} | {psm_dda} | {pg_dia} | {pg_dda} |\n"
    return md


def reader_mode_dict_to_dataframe(reader_mode_dict):
    import pandas as pd
    rows = []
    for package, modes in reader_mode_dict.items():
        rows.append({
            ("PSM Readers", "DIA"): ", ".join(modes["DIA"]["psm"]),
            ("PSM Readers", "DDA"): ", ".join(modes["DDA"]["psm"]),
            ("PG Readers", "DIA"): ", ".join(modes["DIA"]["pg"]),
            ("PG Readers", "DDA"): ", ".join(modes["DDA"]["pg"]),
        })
    df = pd.DataFrame(rows, index=list(reader_mode_dict.keys()))
    df.columns = pd.MultiIndex.from_tuples(df.columns)
    return df


# Generate tables
print(reader_mode_dict_to_latex_table(reader_mode_dict))
print()
print(reader_mode_dict_to_markdown_table(reader_mode_dict))
reader_mode_dict_to_dataframe(reader_mode_dict)

\begin{tabularx}{\textwidth}{l X X X X}
\toprule
 & \multicolumn{2}{c}{PSM readers} & \multicolumn{2}{c}{Protein-group readers} \\
\cmidrule(lr){2-3} \cmidrule(lr){4-5}
Package & DIA & DDA & DIA & DDA \\
\midrule
ProtData & Spectronaut & --- & MaxQuant, DIA-NN, FragPipe, Spectronaut, mzTab & MaxQuant, FragPipe, mzTab \\
\midrule
MSMU & DIA-NN, DELPI & MaxQuant, Sage, FragPipe & --- & --- \\
\midrule
ProteoPy & DIA-NN & --- & --- & --- \\
\midrule
AlphaPeptTools & AlphaDIA, DIA-NN, Spectronaut, OpenSWATH, Spectronaut speclib, MSFragger, Sage & MaxQuant, AlphaPept, MSFragger, Sage, pFind & AlphaDIA, DIA-NN, Spectronaut, FragPipe, DirectLFQ, mzTab & MaxQuant, AlphaPept, FragPipe, DirectLFQ, mzTab, Proteome Discoverer \\
\bottomrule
\end{tabularx}

| Package | PSM-DIA | PSM-DDA | PG-DIA | PG-DDA |
|---|---|---|---|---|
| ProtData | Spectronaut | — | MaxQuant, DIA-NN, FragPipe, Spectronaut, mzTab | MaxQuant, FragPipe, mzTab |
| MSMU | DIA-NN, DELPI | MaxQuant, Sage, FragPipe | — | — |
| Pro

PSM Readers  \
                                                              DIA   
ProtData                                              Spectronaut   
MSMU                                                DIA-NN, DELPI   
ProteoPy                                                   DIA-NN   
AlphaPeptTools  AlphaDIA, DIA-NN, Spectronaut, OpenSWATH, Spec...   

                                                             \
                                                        DDA   
ProtData                                                      
MSMU                               MaxQuant, Sage, FragPipe   
ProteoPy                                                      
AlphaPeptTools  MaxQuant, AlphaPept, MSFragger, Sage, pFind   

                                                       PG Readers  \
                                                              DIA   
ProtData           MaxQuant, DIA-NN, FragPipe, Spectronaut, mzTab   
MSMU                                                                
ProteoPy                                                            
AlphaPeptTools  AlphaDIA, DIA-NN, Spectronaut, FragPipe, Direc...   

                                                                   
                                                              DDA  
ProtData                                MaxQuant, FragPipe, mzTab  
MSMU                                                               
ProteoPy                                                           
AlphaPeptTools  MaxQuant, AlphaPept, FragPipe, DirectLFQ, mzTa...

In [9]:
# Reader dict with simple layout but still retaining DDA/DIA information
# Same column layout as the existing manuscript table (l X X). Each tool name carries a
# superscript derived from reader_mode_dict:
#   1   -> DIA only
#   2   -> DDA only
#   1,2 -> both
# The caption legend additionally enumerates every tool/format that appears under each
# mode — a glance-able glossary of the search engines and exchange formats represented.


def _mode_marks(package, level, tool, reader_mode_dict, dia_mark, dda_mark, separator):
    marks = []
    modes = reader_mode_dict.get(package, {})
    if tool in modes.get("DIA", {}).get(level, []):
        marks.append(dia_mark)
    if tool in modes.get("DDA", {}).get(level, []):
        marks.append(dda_mark)
    return separator.join(marks)


def _collect_tools_by_mode(reader_mode_dict, mode, tool_name_overrides=None):
    """Alphabetically-sorted (case-insensitive) unique tool display names for a given mode."""
    tool_name_overrides = tool_name_overrides or {}
    tools = set()
    for modes in reader_mode_dict.values():
        for level_tools in modes.get(mode, {}).values():
            for tool in level_tools:
                tools.add(tool_name_overrides.get(tool, tool))
    return sorted(tools, key=str.lower)


def reader_dict_to_latex_table_with_modes(
    reader_dict,
    reader_mode_dict,
    citations=None,
    package_name_overrides=None,
    tool_name_overrides=None,
    caption=None,
    label="tab:readers",
    empty_placeholder="---",
    dia_mark="1",
    dda_mark="2",
):
    """Drop-in LaTeX block matching the manuscript's existing reader table, with mode superscripts."""
    citations = citations or {}
    package_name_overrides = package_name_overrides or {}
    tool_name_overrides = tool_name_overrides or {}

    dia_tools = _collect_tools_by_mode(reader_mode_dict, "DIA", tool_name_overrides)
    dda_tools = _collect_tools_by_mode(reader_mode_dict, "DDA", tool_name_overrides)
    default_caption = (
        "Reader functions implemented across proteomics packages built on the AnnData/MuData "
        "ecosystem. PSM-level readers ingest precursor- or peptide-level search outputs (features "
        "are peptide identifiers); protein-group (PG) readers ingest protein-group-level matrices "
        "(features are protein identifiers). Superscripts denote acquisition mode support: "
        f"\\textsuperscript{{{dia_mark}}} DIA ({', '.join(dia_tools)}); "
        f"\\textsuperscript{{{dda_mark}}} DDA ({', '.join(dda_tools)})."
    )
    caption = caption if caption is not None else default_caption

    def annotate(package, level, tool):
        marks = _mode_marks(package, level, tool, reader_mode_dict, dia_mark, dda_mark, ",")
        display = tool_name_overrides.get(tool, tool)
        return f"{display}\\textsuperscript{{{marks}}}" if marks else display

    lines = [
        "\\begin{table}[ht]",
        "\\centering",
        f"\\caption{{{caption}}}\\label{{{label}}}",
        "\\begin{tabularx}{\\textwidth}{l X X}",
        "\\toprule",
        "Package & PSM readers & Protein-group readers \\\\",
        "\\midrule",
    ]
    packages = list(reader_dict.items())
    for i, (package, readers) in enumerate(packages):
        display_pkg = package_name_overrides.get(package, package)
        cite = f"\\cite{{{citations[package]}}}" if package in citations else ""
        psm = ", ".join(annotate(package, "psm", t) for t in readers["psm"]) or empty_placeholder
        pg = ", ".join(annotate(package, "pg", t) for t in readers["pg"]) or empty_placeholder
        lines.append(f"{display_pkg}{cite} & {psm} & {pg} \\\\")
        if i < len(packages) - 1:
            lines.append("\\midrule")
    lines.append("\\bottomrule")
    lines.append("\\end{tabularx}")
    lines.append("\\end{table}")
    return "\n".join(lines)


def reader_dict_to_markdown_table_with_modes(
    reader_dict,
    reader_mode_dict,
    empty_placeholder="—",
    dia_mark="¹",
    dda_mark="²",
):
    def annotate(package, level, tool):
        marks = _mode_marks(package, level, tool, reader_mode_dict, dia_mark, dda_mark, "")
        return tool + marks

    md = "| Package | PSM Readers | PG Readers |\n"
    md += "|---|---|---|\n"
    for package, readers in reader_dict.items():
        psm = ", ".join(annotate(package, "psm", t) for t in readers["psm"]) or empty_placeholder
        pg = ", ".join(annotate(package, "pg", t) for t in readers["pg"]) or empty_placeholder
        md += f"| {package} | {psm} | {pg} |\n"
    return md


def reader_dict_to_dataframe_with_modes(
    reader_dict,
    reader_mode_dict,
    dia_mark="¹",
    dda_mark="²",
):
    import pandas as pd
    def annotate(package, level, tool):
        marks = _mode_marks(package, level, tool, reader_mode_dict, dia_mark, dda_mark, "")
        return tool + marks

    data = []
    for package, readers in reader_dict.items():
        psm = ", ".join(annotate(package, "psm", t) for t in readers["psm"])
        pg = ", ".join(annotate(package, "pg", t) for t in readers["pg"])
        data.append({"Package": package, "PSM Readers": psm, "PG Readers": pg})
    return pd.DataFrame(data)


# Manuscript-specific name and citation maps
citations = {
    "ProtData": "Frank.2025",
    "MSMU": "Choi.2026",
    "ProteoPy": "Fichtner.2026",
}

package_name_overrides = {
    "AlphaPeptTools": "alphapepttools",
}

tool_name_overrides = {
    "AlphaDIA": "AlphaDia",
    "Proteome Discoverer": "ProteomeDiscoverer",
}


# Generate tables
print(reader_dict_to_latex_table_with_modes(
    reader_dict,
    reader_mode_dict,
    citations=citations,
    package_name_overrides=package_name_overrides,
    tool_name_overrides=tool_name_overrides,
))
print()
print(reader_dict_to_markdown_table_with_modes(reader_dict, reader_mode_dict))
reader_dict_to_dataframe_with_modes(reader_dict, reader_mode_dict)

\begin{table}[ht]
\centering
\caption{Reader functions implemented across proteomics packages built on the AnnData/MuData ecosystem. PSM-level readers ingest precursor- or peptide-level search outputs (features are peptide identifiers); protein-group (PG) readers ingest protein-group-level matrices (features are protein identifiers). Superscripts denote acquisition mode support: \textsuperscript{1} DIA (AlphaDia, DELPI, DIA-NN, DirectLFQ, FragPipe, MaxQuant, MSFragger, mzTab, OpenSWATH, Sage, Spectronaut, Spectronaut speclib); \textsuperscript{2} DDA (AlphaPept, DirectLFQ, FragPipe, MaxQuant, MSFragger, mzTab, pFind, ProteomeDiscoverer, Sage).}\label{tab:readers}
\begin{tabularx}{\textwidth}{l X X}
\toprule
Package & PSM readers & Protein-group readers \\
\midrule
ProtData\cite{Frank.2025} & Spectronaut\textsuperscript{1} & MaxQuant\textsuperscript{1,2}, DIA-NN\textsuperscript{1}, FragPipe\textsuperscript{1,2}, Spectronaut\textsuperscript{1}, mzTab\textsuperscript{1,2} \\
\midrule
MSMU

,Package,PSM Readers,PG Readers
0,ProtData,Spectronaut¹,"MaxQuant¹², DIA-NN¹, FragPipe¹², Spectronaut¹,..."
1,MSMU,"Sage², MaxQuant², FragPipe², DIA-NN¹, DELPI¹",
2,ProteoPy,DIA-NN¹,
3,AlphaPeptTools,"AlphaDIA¹, MaxQuant², DIA-NN¹, AlphaPept², Spe...","AlphaDIA¹, MaxQuant², DIA-NN¹, AlphaPept², Spe..."
